# 1. Conda environment and a first COF build

Language: **English** | [中文](../cn/01_environment_and_first_build.ipynb)

This notebook creates the repository's `cofkit` Conda environment, registers it as a Jupyter kernel, and builds a TAPB–TFB imine COF on the `hcb` topology. Run the notebook from a clone of the repository.

The executable cells use the CLI through shell commands. Commented Python cells immediately below the main operations show the equivalent API workflow without executing it.

## Tutorial series

1. **Environment and first build** (this notebook)
2. [Topology, connectivity, and linkage examples](02_topologies_connectivities_and_linkages.ipynb)
3. [Pore analysis with Zeo++](03_zeopp_pore_analysis.ipynb)

Launch Jupyter from the repository root so that all relative paths below resolve consistently. Shell cells use `conda run -n cofkit`, so creating or activating the notebook kernel does not need to persist across cells.

## Inspect the checked-in environment

The first line of `environment.yml` fixes the environment name as `cofkit`. The same name is used by this entire tutorial series.

In [ ]:
%%bash
set -euo pipefail
ROOT="$(git rev-parse --show-toplevel)"
cd "$ROOT"
printf 'Repository root: %s\n\n' "$ROOT"
sed -n '1,24p' environment.yml

In [ ]:
# Python equivalent (commented out): inspect the environment name.
# from pathlib import Path
# environment_name = next(
#     line.split(":", 1)[1].strip()
#     for line in Path("environment.yml").read_text().splitlines()
#     if line.startswith("name:")
# )
# print(environment_name)

## Create or update the environment

This cell is safe to rerun: it creates `cofkit` when absent and otherwise updates it from `environment.yml`. It then installs the local package without replacing Conda-resolved dependencies and registers a `Python (cofkit)` kernel for Notebooks 2 and 3.

> Environment resolution and the Open Babel installation can take several minutes on a first run.

In [ ]:
%%bash
set -euo pipefail
ROOT="$(git rev-parse --show-toplevel)"
cd "$ROOT"
command -v conda >/dev/null || { echo 'Conda is required. Install Miniforge or Miniconda, then rerun this cell.' >&2; exit 1; }
if conda env list | awk '{print $1}' | grep -Fxq cofkit; then
  conda env update --name cofkit --file environment.yml --prune
else
  conda env create --file environment.yml
fi
conda run -n cofkit python -m pip install --no-deps .
conda install --name cofkit --channel conda-forge --yes ipykernel
conda run -n cofkit python -m ipykernel install --user --name cofkit --display-name 'Python (cofkit)'

In [ ]:
# There is intentionally no Python API equivalent for creating a Conda environment.
# Conda and pip are environment-management tools, so the shell commands above are canonical.

If this notebook was opened with another kernel, you may now select **Python (cofkit)** from Jupyter's kernel menu. The remaining executable cells explicitly use `conda run -n cofkit`, so switching is optional here. Notebooks 2 and 3 declare `cofkit` as their default kernel.

## Verify the installation

In [ ]:
%%bash
set -euo pipefail
ROOT="$(git rev-parse --show-toplevel)"
cd "$ROOT"
conda run -n cofkit cofkit --version
conda run -n cofkit python -c "import cofkit; print('Python import:', cofkit.__version__)"
conda run -n cofkit cofkit build list-templates

In [ ]:
# Python equivalent (commented out): verify that the package imports.
# import cofkit
# print(cofkit.__version__)
#
# from cofkit import ReactionLibrary
# library = ReactionLibrary.builtin()
# print(library)

## Build TAPB–TFB on `hcb`

TAPB and TFB each expose three reactive groups, giving a 3+3 connection combination. The `imine_bridge` template joins amines to aldehydes, while `--topology hcb` makes the topology choice explicit and reproducible.

The command writes `summary.json` and places exported CIFs below `out/tutorial_01_first_build/cifs/`, grouped by coarse validation class.

In [ ]:
%%bash
set -euo pipefail
ROOT="$(git rev-parse --show-toplevel)"
cd "$ROOT"
conda run -n cofkit cofkit build single-pair \
  --template-id imine_bridge \
  --first-smiles 'C1=CC(=CC=C1C2=CC(=CC(=C2)C3=CC=C(C=C3)N)C4=CC=C(C=C4)N)N' \
  --second-smiles 'C1=C(C=C(C=C1C=O)C=O)C=O' \
  --first-id tapb \
  --second-id tfb \
  --first-motif-kind amine \
  --second-motif-kind aldehyde \
  --topology hcb \
  --target-dimensionality 2D \
  --output-dir out/tutorial_01_first_build

In [ ]:
# Python API equivalent (commented out).
# from pathlib import Path
# from cofkit import BatchGenerationConfig, BatchStructureGenerator, build_rdkit_monomer
#
# tapb = build_rdkit_monomer(
#     "tapb",
#     "TAPB",
#     "C1=CC(=CC=C1C2=CC(=CC(=C2)C3=CC=C(C=C3)N)C4=CC=C(C=C4)N)N",
#     "amine",
#     num_conformers=4,
# )
# tfb = build_rdkit_monomer(
#     "tfb",
#     "TFB",
#     "C1=C(C=C(C=C1C=O)C=O)C=O",
#     "aldehyde",
#     num_conformers=4,
# )
# generator = BatchStructureGenerator(
#     BatchGenerationConfig(
#         allowed_reactions=("imine_bridge",),
#         target_dimensionality="2D",
#         topology_ids=("hcb",),
#         rdkit_num_conformers=4,
#         max_workers=1,
#         write_cif=True,
#     )
# )
# summaries, candidates, attempted = generator.generate_monomer_pair_candidates(
#     tapb,
#     tfb,
#     pair_id="tapb__tfb",
#     out_dir=Path("out/tutorial_01_first_build_python/cifs"),
# )
# print("attempted:", attempted)
# for summary in summaries:
#     print(summary.topology_id, summary.status, summary.score, summary.cif_path)

## Inspect the build output

The CIF may appear under `valid`, `warning`, or `needs_optimization`; that directory is a coarse geometry classification, not a different chemistry. Notebook 3 searches all of these subdirectories rather than assuming one class.

In [ ]:
%%bash
set -euo pipefail
ROOT="$(git rev-parse --show-toplevel)"
cd "$ROOT"
printf '%s\n' '--- summary.json (first 100 lines) ---'
sed -n '1,100p' out/tutorial_01_first_build/summary.json
printf '%s\n' '--- exported CIF files ---'
find out/tutorial_01_first_build/cifs -type f -name '*.cif' -print | sort

In [ ]:
# Python equivalent (commented out): inspect the structured report and locate CIFs.
# import json
# from pathlib import Path
# output_dir = Path("out/tutorial_01_first_build")
# summary = json.loads((output_dir / "summary.json").read_text())
# print(summary["successful_structures"], summary["cifs_written"])
# print(*sorted((output_dir / "cifs").rglob("*.cif")), sep="\n")

## What to keep

- `summary.json` records the requested chemistry, detected motif counts, attempted structures, scores, validation flags, and CIF paths.
- Each CIF starts with a `# COFid:` comment that captures monomers, topology, and linkage.
- These are initial generated structures. A warning or `needs_optimization` result should be optimized and revalidated before simulation or publication.
- Keep `out/tutorial_01_first_build/` in place: Notebook 3 uses its CIF as the default Zeo++ input.

Continue with [Notebook 2](02_topologies_connectivities_and_linkages.ipynb) to compare a small set of other build choices.